<a href="https://colab.research.google.com/github/tamara-kostova/MSc_Thesis_Neuroimaging/blob/master/11_medgemma_full_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Medgemma 1.5 - full evaluation

## Imports and packages

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [30]:
!pip install -q pillow tqdm pandas requests

import os
import json
import base64
import requests
import pandas as pd

from pathlib import Path
from io import BytesIO
from PIL import Image
from tqdm import tqdm
from datetime import datetime

## Config

In [44]:
BASE_DIR = "/content/drive/MyDrive/MSc_Thesis_Neuroimaging"
RAW_DIR = f"{BASE_DIR}/data/raw data"
SPLITS_DIR = f"{BASE_DIR}/subsets"
RESULTS_DIR = f"{BASE_DIR}/results/medgemma"

os.makedirs(RESULTS_DIR, exist_ok=True)

with open(f"{BASE_DIR}/neurorad_prompt_trim_5classes_refined.txt") as f:
    SYSTEM_PROMPT = f.read()

FEW_SHOT_EXAMPLES_CSV = f"{BASE_DIR}/few_shot_examples.csv"

MODEL_NAME = "google/medgemma-1.5-4b-it"
VLLM_ENDPOINT = "http://<ENDPOINTS>"

PHASES = {
    "Phase_1_ZeroShot": {
        "splits": ["subset_1.csv", "subset_2.csv", "subset_3.csv"],
        "few_shot": False
    },
    "Phase_2_ZeroShot": {
        "splits": ["subset_1.csv", "subset_2.csv", "subset_0_25.csv"],
        "few_shot": False
    },
    "Phase_3_FewShot": {
        "splits": ["subset_0_25_7.csv"],
        "few_shot": True
    }
}


## Helpers

In [51]:
def find_image_path(base_path, filename):
    """Search for image file in common locations for all datasets"""
    candidate_paths = [
        base_path / filename,
        base_path / "2" / filename,
        base_path.parent / filename,
    ]

    if "figshare" in str(base_path).lower():
        figshare_root = base_path.parent.parent

        for dataset_folder in figshare_root.glob("brainTumorData*"):
            candidates = [
                dataset_folder / filename,
                dataset_folder / "2" / filename,
            ]
            candidate_paths.extend(candidates)

    unique_candidates = list(set(candidate_paths))
    for candidate in unique_candidates:
        if candidate.exists():
            return candidate

    return None


In [48]:
from io import BytesIO
from PIL import Image
import base64

def encode_image(file_path_str):
    file_path = Path(file_path_str)

    if file_path.exists():
        img_path = file_path
    else:
        filename = file_path.name
        base_path = Path(RAW_DIR) / file_path.parent
        img_path = find_image_path(base_path, filename)

        if img_path is None:
            raise FileNotFoundError(f"Image not found: {file_path_str}")

    img = Image.open(img_path).convert("RGB")
    buf = BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8"), str(img_path)

In [5]:
def parse_vllm_response(resp):
    content = resp.json()["choices"][0]["message"]["content"]

    start = content.find('{')
    end = content.rfind('}') + 1
    if start >= 0 and end > start:
        json_str = content[start:end]
        return json.loads(json_str)
    return {}

In [6]:
import re
import json

def extract_json(text):
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None

In [7]:
def load_few_shot_messages():
    df = pd.read_csv(FEW_SHOT_EXAMPLES_CSV)

    messages = []
    for _, row in df.iterrows():
        img_path = Path(RAW_DIR) / row["file_path"]
        img_b64 = encode_image(img_path)

        messages.append({
            "role": "user",
            "content": [
                {"type": "text", "text": "Analyze this scan."},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
            ]
        })

        messages.append({
            "role": "assistant",
            "content": json.dumps({
                "modality": row["modality"] if pd.notna(row["modality"]) else None,
                "specialized_sequence": row["modality_subtype"] if pd.notna(row["modality_subtype"]) else None,
                "plane": row["plane"] if pd.notna(row["plane"]) else None,
                "diagnosis_name": row["class"],
                "diagnosis_detailed": row["subclass"] if pd.notna(row["subclass"]) else None
            })
        })

    return messages


In [8]:
def build_payload(image_b64, few_shot_messages=None):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if few_shot_messages:
        messages.extend(few_shot_messages)

    messages.append({
        "role": "user",
        "content": [
            {"type": "text", "text": "Analyze this scan."},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}}
        ]
    })

    return {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": 0,
        "top_p": 1.0,
        "max_tokens": 300
    }


In [20]:
def create_row(image_path, row_data, prediction, api_response, phase_name, row_index, split_name):
    """Create row in exact format specified"""
    rel_path = str(image_path.relative_to(BASE_DIR))

    metadata = {
        "dataset": row_data.get("dataset", ""),
        "class": row_data.get("class", ""),
        "subclass": row_data.get("subclass", ""),
        "modality": row_data.get("modality", ""),
        "axial_plane": row_data.get("plane", "")
    }

    return {
        "timestamp": datetime.now().isoformat(),
        "experiment_id": f"medgemma_{phase_name}_{row_index:06d}",
        "api_response": api_response,
        "input": {
            "image_path": rel_path,
            "model_requested": MODEL_NAME,
            "system_prompt_type": "neurorad_prompt_trim_5classes_refined",
            "metadata": metadata
        },
        "output": {
            "parsed_response": prediction,
            "status": "success" if prediction else "error",
            "error": None
        },
        "processing": {
            "row_index": row_index,
            "parameters": {
                "temperature": 0,
                "seed": 42
            },
            "phase": phase_name,
            "split": split_name,
            "few_shot": "yes" if "FewShot" in phase_name else "no"
        }
    }


# Run phases

In [52]:
def run_phase(phase_name, phase_cfg):
    phase_dir = Path(RESULTS_DIR) / phase_name
    phase_dir.mkdir(parents=True, exist_ok=True)

    few_shot_messages = load_few_shot_messages() if phase_cfg["few_shot"] else None
    all_rows = []

    global_row_index = 0

    for split_idx, split_csv in enumerate(phase_cfg["splits"]):
        split_name = Path(split_csv).stem
        split_dir = phase_dir / split_name
        split_dir.mkdir(exist_ok=True)

        split_rows = []
        df = pd.read_csv(Path(SPLITS_DIR) / split_csv)
        print(f"▶ {phase_name} / {split_name}: {len(df)} images")

        for row_idx, row in tqdm(df.iterrows(), total=len(df)):
            img_path = Path(RAW_DIR) / row["file_path"]
            out_file = split_dir / f"{img_path.stem}.json"

            if out_file.exists():
                try:
                    with open(out_file, 'r') as f:
                        row_data = json.load(f)
                    split_rows.append(row_data)
                    all_rows.append(row_data)
                except:
                    pass
                continue

            try:
                img_b64, actual_img_path = encode_image(img_path)
                payload = build_payload(img_b64, few_shot_messages)

                response = requests.post(VLLM_ENDPOINT, json=payload, timeout=60)
                api_resp = response.json() if response.status_code == 200 else {"error": response.text}

                parsed = parse_vllm_response(response) if response.status_code == 200 else {}

                row_data = create_row(
                    img_path, row.to_dict(),
                    parsed, api_resp,
                    phase_name, global_row_index, split_name
                )
                global_row_index += 1

                with open(out_file, "w") as f:
                    json.dump(row_data, f, indent=2)

                split_rows.append(row_data)
                all_rows.append(row_data)

            except Exception as e:
                error_row = create_row(
                    img_path, row.to_dict(),
                    None, {"error": str(e)},
                    phase_name, global_row_index, split_name
                )
                error_row["output"]["status"] = "error"
                error_row["output"]["error"] = str(e)

                with open(out_file, "w") as f:
                    json.dump(error_row, f, indent=2)

                split_rows.append(error_row)
                all_rows.append(error_row)

        split_summary_file = split_dir / "summary.json"
        with open(split_summary_file, "w") as f:
            json.dump(split_rows, f, indent=2)
        print(f"  └─ Saved {len(split_rows)} rows to {split_summary_file}")

    phase_summary_file = phase_dir / "phase_summary.json"
    with open(phase_summary_file, "w") as f:
        json.dump(all_rows, f, indent=2)
    print(f"{phase_name}: Saved {len(all_rows)} total rows to {phase_summary_file}")

In [50]:
for phase_name, cfg in PHASES.items():
    run_phase(phase_name, cfg)


▶ Phase_1_ZeroShot / subset_1: 2950 images


100%|██████████| 3/3 [00:00<00:00, 223.38it/s]


  └─ Saved 3 rows to /content/drive/MyDrive/MSc_Thesis_Neuroimaging/results/medgemma/Phase_1_ZeroShot/subset_1/summary.json
▶ Phase_1_ZeroShot / subset_2: 2950 images


100%|██████████| 3/3 [00:00<00:00, 170.84it/s]


  └─ Saved 3 rows to /content/drive/MyDrive/MSc_Thesis_Neuroimaging/results/medgemma/Phase_1_ZeroShot/subset_2/summary.json
▶ Phase_1_ZeroShot / subset_3: 2950 images


...

  └─ Saved 3 rows to /content/drive/MyDrive/MSc_Thesis_Neuroimaging/results/medgemma/Phase_2_ZeroShot/subset_2/summary.json
▶ Phase_2_ZeroShot / subset_0_25: 7377 images


100%|██████████| 3/3 [00:00<00:00, 206.54it/s]


  └─ Saved 3 rows to /content/drive/MyDrive/MSc_Thesis_Neuroimaging/results/medgemma/Phase_2_ZeroShot/subset_0_25/summary.json
Phase_2_ZeroShot: Saved 9 total rows to /content/drive/MyDrive/MSc_Thesis_Neuroimaging/results/medgemma/Phase_2_ZeroShot/phase_summary.json
Figshare search for 2828.jpg in: /content/drive/MyDrive/MSc_Thesis_Neuroimaging/data/raw data/figshare
✅ Found 2828.jpg at: /content/drive/MyDrive/MSc_Thesis_Neuroimaging/data/raw data/figshare/brainTumorDataPublic_2299-3064/2828.jpg
▶ Phase_3_FewShot / subset_0_25_7: 7377 images


100%|██████████| 3/3 [00:02<00:00,  1.23it/s]

  └─ Saved 3 rows to /content/drive/MyDrive/MSc_Thesis_Neuroimaging/results/medgemma/Phase_3_FewShot/subset_0_25_7/summary.json
Phase_3_FewShot: Saved 3 total rows to /content/drive/MyDrive/MSc_Thesis_Neuroimaging/results/medgemma/Phase_3_FewShot/phase_summary.json
